## 2. Linear Regression Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json
import numpy as np

from src.dataloader import load_arrays, load_cgm_stats, load_participant_ids
from src.metrics import evaluate, format_results_table, evaluate_per_participant
from src.models.classical_baseline import LinearRegressionPredictor, HORIZONS

MODEL_NAME = 'LinearRegression'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'linear_regression'
RESULTS_PATH = RESULTS_DIR / 'lr_results.json'
VAL_PARTICIPANT_SUMMARY_PATH = RESULTS_DIR / 'lr_val_participant_stability_summary.json'
PARTICIPANT_SUMMARY_PATH = RESULTS_DIR / 'lr_participant_stability_summary.json'

In [3]:
stats = load_cgm_stats()
X_train, y_train = load_arrays('train')
X_val, y_val = load_arrays('val')
X_test, y_test = load_arrays('test')

val_participant_ids = load_participant_ids('val')
test_participant_ids = load_participant_ids('test')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('train set shape: ', X_train.shape, '\nvaluation set shape: ', X_val.shape, '\ntest set shape: ', X_test.shape)
print('number of validation participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

train set shape:  (83532, 24) 
valuation set shape:  (16739, 24) 
test set shape:  (16842, 24)
number of validation participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]


In [4]:
model = LinearRegressionPredictor().fit(X_train, y_train)

In [5]:
val_results = evaluate(model.predict(X_val), y_val, stats['cgm_mean'], stats['cgm_std'])
test_results = evaluate(model.predict(X_test), y_test, stats['cgm_mean'], stats['cgm_std'])
val_results_per_participant = evaluate_per_participant(
    model.predict(X_val), y_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])
test_results_per_participant = evaluate_per_participant(
    model.predict(X_test), y_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

print('=== Validation ===')
print(format_results_table(val_results))

=== Validation ===
Horizon (min) |     RMSE |      MAE |    gRMSE |  Zone A % | Zone A+B %
-----------------------------------------------------------------------
           15 |    11.43 |     7.74 |    12.40 |     96.15 |      99.78
           30 |    18.39 |    12.42 |    20.73 |     88.49 |      99.38
           60 |    25.86 |    17.42 |    30.66 |     80.31 |      98.72
           90 |    30.25 |    20.53 |    36.63 |     74.88 |      98.17
          120 |    32.91 |    22.62 |    40.26 |     71.60 |      97.81


In [6]:
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_participant_summary(pooled_results, results_per_participant, participant_ids):
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            mean = _extract(pooled_results[h], m)

            values = np.array([_extract(results_per_participant[pid][h], m)
                                for pid in participant_ids])
            summary[h][m] = {'mean': float(mean), 'sd1_across_seeds': 0.0,
                              'sd2_across_participants': float(values.std(ddof=1))}
    return summary

val_participant_summary = build_participant_summary(
    val_results, val_results_per_participant, VAL_PARTICIPANT_IDS)
test_participant_summary = build_participant_summary(
    test_results, test_results_per_participant, PARTICIPANT_IDS)

def _print_summary(split_label, participant_ids, summary):
    print(f'=== {MODEL_NAME} {split_label} set: mean +/- sd1 (sd2) -- sd1=0.00 (no seed), '
          f'sd2 across {len(participant_ids)} participants ===')
    for h in HORIZONS:
        s = summary[h]
        print(f"{h:>3}-min:  "
              f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
              f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
              f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
              f"({s['mae']['sd2_across_participants']:.2f})   "
              f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
              f"({s['grmse']['sd2_across_participants']:.2f})   "
              f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
              f"({s['zone_a']['sd2_across_participants']:.2f})%   "
              f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
              f"({s['zone_ab']['sd2_across_participants']:.2f})%")

_print_summary('validation', VAL_PARTICIPANT_IDS, val_participant_summary)
print()
_print_summary('test', PARTICIPANT_IDS, test_participant_summary)

with open(VAL_PARTICIPANT_SUMMARY_PATH, 'w') as f:
    json.dump({'model': MODEL_NAME, 'participant_ids': VAL_PARTICIPANT_IDS,
               'per_horizon_summary': val_participant_summary}, f, indent=2)

with open(PARTICIPANT_SUMMARY_PATH, 'w') as f:
    json.dump({'model': MODEL_NAME, 'participant_ids': PARTICIPANT_IDS,
               'per_horizon_summary': test_participant_summary}, f, indent=2)

print(f'\nsaved validation (selection) summary to {VAL_PARTICIPANT_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {PARTICIPANT_SUMMARY_PATH}')

=== LinearRegression validation set: mean +/- sd1 (sd2) -- sd1=0.00 (no seed), sd2 across 6 participants ===
 15-min:  RMSE = 11.43 +/- 0.00 (2.45) mg/dL   MAE = 7.74 +/- 0.00 (1.63)   gRMSE = 12.40 +/- 0.00 (3.32)   Zone A = 96.15 +/- 0.00 (3.72)%   Zone A+B = 99.78 +/- 0.00 (0.40)%
 30-min:  RMSE = 18.39 +/- 0.00 (4.05) mg/dL   MAE = 12.42 +/- 0.00 (2.61)   gRMSE = 20.73 +/- 0.00 (6.12)   Zone A = 88.49 +/- 0.00 (6.49)%   Zone A+B = 99.38 +/- 0.00 (0.80)%
 60-min:  RMSE = 25.86 +/- 0.00 (7.78) mg/dL   MAE = 17.42 +/- 0.00 (4.98)   gRMSE = 30.66 +/- 0.00 (11.94)   Zone A = 80.31 +/- 0.00 (5.67)%   Zone A+B = 98.72 +/- 0.00 (1.44)%
 90-min:  RMSE = 30.25 +/- 0.00 (10.82) mg/dL   MAE = 20.53 +/- 0.00 (7.19)   gRMSE = 36.63 +/- 0.00 (16.17)   Zone A = 74.88 +/- 0.00 (7.71)%   Zone A+B = 98.17 +/- 0.00 (2.05)%
120-min:  RMSE = 32.91 +/- 0.00 (12.62) mg/dL   MAE = 22.62 +/- 0.00 (8.67)   gRMSE = 40.26 +/- 0.00 (18.85)   Zone A = 71.60 +/- 0.00 (9.09)%   Zone A+B = 97.81 +/- 0.00 (2.57)%

=

In [7]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seed': None,  # deterministic, no seed applies
        'val_results': val_results,
        'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }, f, indent=2)

print(f'saved to {RESULTS_PATH}')

saved to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/linear_regression/lr_results.json
